# 🏛️ Tecnocracia: Banco de Pruebas y Evaluación Multiagente
En este notebook puedes probar interactivamente la deliberación del gobierno tecnocrático y medir:
1. **Telemetría y Rendimiento:** Tiempo de respuesta (latencia) y consumo de tokens por ministro.
2. **Resolución y Publicación:** El dictamen consensuado y el hilo para redes sociales.
3. **Evaluación de Calidad:** Una rúbrica que audita el rigor cuantitativo, la viabilidad presupuestaria y la transparencia.

In [ ]:
# Celda 1: Carga de configuración y cliente Gemini
import os
import sys
import time
import json
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Añadir directorio raíz a sys.path
root_dir = Path.cwd().parent if Path.cwd().name == "primer_ministro" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

load_dotenv(root_dir / ".env")
api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

from ministros import ministro_economia, ministro_educacion, ministro_interior
print("✅ Entorno y ministros cargados correctamente.")

In [ ]:
# Celda 2: Define el problema regional que quieres someter al gobierno
problema_regional = """
DESAFÍO REGIONAL:
- El desempleo juvenil ha subido al 28% y faltan 15.000 profesionales técnicos.
- El presupuesto extraordinario disponible es de 200 millones de euros.
- Creciente descontento ciudadano en redes sociales exigiendo soluciones inmediatas y transparentes.
"""
print("📋 Problema planteado:")
print(problema_regional)

In [ ]:
# Celda 3: Función de ejecución con captura de telemetría (tiempo y tokens)
registro_metricas = []

def consultar_agente(nombre, instruccion, contexto):
    t0 = time.perf_counter()
    res = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"{instruccion}\n\nSITUACIÓN:\n{contexto}"
    )
    duracion = time.perf_counter() - t0
    tokens_in = getattr(res.usage_metadata, 'prompt_token_count', 0)
    tokens_out = getattr(res.usage_metadata, 'candidates_token_count', 0)
    
    metrica = {
        'agente': nombre,
        'duracion': duracion,
        'tokens_in': tokens_in,
        'tokens_out': tokens_out,
        'tokens_total': tokens_in + tokens_out,
        'texto': res.text
    }
    registro_metricas.append(metrica)
    print(f"⏱️ {nombre}: {duracion:.2f}s | 🏷️ Tokens: {tokens_in + tokens_out} (in: {tokens_in}, out: {tokens_out})")
    return res.text

In [ ]:
# Celda 4: Deliberación del Consejo de Ministros
print("1️⃣ Consultando al Ministro de Economía...")
propuesta_economia = consultar_agente("Economía", ministro_economia.instruction, problema_regional)

print("\n2️⃣ Consultando al Ministro de Educación...")
contexto_edu = f"{problema_regional}\n\nPropuesta de Economía:\n{propuesta_economia}"
propuesta_educacion = consultar_agente("Educación", ministro_educacion.instruction, contexto_edu)

print("\n3️⃣ Consultando al Ministro de Interior...")
contexto_int = f"{problema_regional}\n\nPropuesta de Economía:\n{propuesta_economia[:400]}\n\nPropuesta de Educación:\n{propuesta_educacion[:400]}"
propuesta_interior = consultar_agente("Interior", ministro_interior.instruction, contexto_int)

In [ ]:
# Celda 5: Síntesis del Primer Ministro y Publicación en Redes Sociales
instruccion_pm = """
Eres el Primer Ministro Tecnocrático.
Arbitra las propuestas de tus ministros y genera:
1. DICTAMEN FINAL TECNOCRÁTICO: Plan con medidas, presupuestos y KPIs de éxito.
2. HILO DE REDES SOCIALES (Twitter/X): 4 publicaciones numeradas explicando las medidas con datos rigurosos a la ciudadanía.
"""

debate_total = f"PROBLEMA:\n{problema_regional}\n\nECONOMÍA:\n{propuesta_economia}\n\nEDUCACIÓN:\n{propuesta_educacion}\n\nINTERIOR:\n{propuesta_interior}"

print("👑 Sintetizando resolución del Primer Ministro...")
resolucion_final = consultar_agente("Primer Ministro", instruccion_pm, debate_total)
print("\n" + "="*60)
print(resolucion_final)

In [ ]:
# Celda 6: Auditoría Automática con Rúbrica Tecnocrática y Métricas Finales
prompt_evaluacion = f"""
Eres un Auditor Científico Independiente de Políticas Públicas.
Evalúa de 1 a 10 con una justificación la resolución del gobierno tecnocrático:

PROBLEMA: {problema_regional}
RESOLUCIÓN: {resolucion_final}

Criterios:
1. rigor_tecnico: Uso de métricas y datos cuantificables.
2. viabilidad_presupuestaria: Realismo fiscal y cálculo de retorno.
3. cohesion_interministerial: Integración y balance entre ministerios.
4. transparencia_comunicacion: Claridad pedagógica para los ciudadanos.

Responde en JSON: {{"rigor_tecnico": {{"puntuacion": 8, "motivo": "..."}}, "viabilidad_presupuestaria": {{"puntuacion": 9, "motivo": "..."}}, "cohesion_interministerial": {{"puntuacion": 8, "motivo": "..."}}, "transparencia_comunicacion": {{"puntuacion": 9, "motivo": "..."}}, "nota_media": 8.5, "veredicto": "Aprobado/Excelente"}}
"""

t0 = time.perf_counter()
res_eval = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt_evaluacion,
    config={"response_mime_type": "application/json"}
)
t_eval = time.perf_counter() - t0

print("📊 --- RESUMEN DE TELEMETRÍA ---")
tiempo_total = sum(m['duracion'] for m in registro_metricas) + t_eval
tokens_totales = sum(m['tokens_total'] for m in registro_metricas) + getattr(res_eval.usage_metadata, 'total_token_count', 0)
print(f"Tiempo Total: {tiempo_total:.2f}s | Tokens Consumidos: {tokens_totales} | Coste: $0.00 (Nivel Gratuito)")

print("\n🎯 --- RÚBRICA DE EVALUACIÓN TECNOCRÁTICA ---")
print(json.dumps(json.loads(res_eval.text), indent=2, ensure_ascii=False))